# Chinese -> Vietnamese Translation Notebook (Kaggle-Fixed Version)

This notebook is a stronger, cleaner upgrade over the provided baseline, with extra fixes for Kaggle execution:

- **word-level tokenization** that matches the dataset's existing whitespace segmentation
- **Transformer encoder-decoder** instead of a small GRU seq2seq
- **label smoothing + AdamW + OneCycleLR + AMP**
- **robust Kaggle path handling**
- **GPU compatibility fallback** if Kaggle assigns an incompatible accelerator
- **beam-search decoding fix** for empty-candidate edge cases
- **submission.csv + submission.zip** generation in the required format

> If you are on Kaggle, the safest accelerator for this notebook is **GPU T4**.


In [ ]:
# Robust metric import for Kaggle/offline environments
import sys, subprocess, importlib.util

SACREBLEU_AVAILABLE = importlib.util.find_spec('sacrebleu') is not None
if not SACREBLEU_AVAILABLE:
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'sacrebleu'])
        SACREBLEU_AVAILABLE = importlib.util.find_spec('sacrebleu') is not None
    except Exception as e:
        print('Could not install sacrebleu automatically:', repr(e))
        print('Falling back to a local BLEU approximation for validation only.')


In [ ]:
import os
import math
import json
import time
import random
import zipfile
from pathlib import Path
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

try:
    import sacrebleu
except Exception:
    sacrebleu = None

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


def _extract_ngrams(tokens, n):
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]


def fallback_corpus_bleu(hypotheses, references, max_n=4, smooth=1.0):
    # A light local fallback if sacrebleu is unavailable.
    # Works on pre-tokenized whitespace-separated text.
    hyp_tokens = [h.split() for h in hypotheses]
    ref_tokens = [r.split() for r in references]

    p_ns = []
    for n in range(1, max_n + 1):
        clipped = 0
        total = 0
        for hyp, ref in zip(hyp_tokens, ref_tokens):
            hyp_ngrams = Counter(_extract_ngrams(hyp, n))
            ref_ngrams = Counter(_extract_ngrams(ref, n))
            clipped += sum(min(count, ref_ngrams[ng]) for ng, count in hyp_ngrams.items())
            total += sum(hyp_ngrams.values())
        p_n = (clipped + smooth) / (total + smooth)
        p_ns.append(p_n)

    hyp_len = sum(len(x) for x in hyp_tokens)
    ref_len = sum(len(x) for x in ref_tokens)
    if hyp_len == 0:
        return 0.0
    bp = 1.0 if hyp_len > ref_len else math.exp(1 - ref_len / max(hyp_len, 1))
    bleu = bp * math.exp(sum((1 / max_n) * math.log(max(p, 1e-12)) for p in p_ns))
    return bleu * 100.0


In [ ]:
# -----------------------------
# Configuration
# -----------------------------

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

def pick_device():
    if torch.cuda.is_available():
        try:
            major, minor = torch.cuda.get_device_capability(0)
            gpu_name = torch.cuda.get_device_name(0)
            print(f"Detected GPU: {gpu_name} (sm_{major}{minor})")
            if major >= 7:
                print("Using CUDA")
                return torch.device('cuda')
            print("GPU is incompatible with this PyTorch build. Falling back to CPU.")
        except Exception as e:
            print(f"CUDA inspection failed ({e}). Falling back to CPU.")
    else:
        print("CUDA not available. Using CPU.")
    return torch.device('cpu')

DEVICE = pick_device()
USE_AMP = DEVICE.type == 'cuda'
IS_KAGGLE = os.path.exists('/kaggle')

@dataclass
class CFG:
    # data
    valid_ratio: float = 0.08
    max_src_len: int = 64
    max_tgt_len: int = 64
    min_freq_src: int = 1
    min_freq_tgt: int = 1

    # model
    d_model: int = 256
    nhead: int = 8
    num_encoder_layers: int = 4
    num_decoder_layers: int = 4
    dim_feedforward: int = 768
    dropout: float = 0.15

    # training
    batch_size: int = 160 if DEVICE.type == 'cuda' else 48
    epochs: int = 12 if IS_KAGGLE and DEVICE.type == 'cuda' else 6
    lr: float = 3e-4
    weight_decay: float = 1e-4
    label_smoothing: float = 0.1
    grad_clip: float = 1.0
    num_workers: int = 2 if DEVICE.type == 'cuda' else 0
    early_stopping: int = 4 if IS_KAGGLE else 5

    # decoding
    beam_size: int = 2 if IS_KAGGLE else 4
    min_decode_len: int = 1
    max_decode_len: int = 64
    length_penalty_alpha: float = 0.7
    no_repeat_ngram_size: int = 0

    # misc
    save_dir: str = 'artifacts_mt'
    run_name: str = 'zh_vi_transformer_wordlevel_kaggle_fixed'
    smoke_test: bool = False  # set True for a quick correctness run

cfg = CFG()
print('Device:', DEVICE)
print('Kaggle:', IS_KAGGLE)
print('SacreBLEU available:', sacrebleu is not None)
print(json.dumps(asdict(cfg), indent=2))


In [ ]:
# -----------------------------
# Locate / extract data
# -----------------------------

def maybe_extract_zip():
    zip_candidates = [
        Path('task2.zip'),
        Path('/mnt/data/task2.zip'),
        Path('/content/task2.zip'),
    ]
    search_roots = [Path('/kaggle/input'), Path('/kaggle/working')]
    for root in search_roots:
        if root.exists():
            zip_candidates.extend(root.rglob('task2.zip'))

    seen = set()
    for zp in zip_candidates:
        zp = Path(zp)
        if str(zp) in seen:
            continue
        seen.add(str(zp))
        if zp.exists():
            out_dir = Path('./task2_extracted')
            if not out_dir.exists():
                out_dir.mkdir(parents=True, exist_ok=True)
                with zipfile.ZipFile(zp, 'r') as zf:
                    zf.extractall(out_dir)
                print(f'Extracted {zp} -> {out_dir.resolve()}')
            return out_dir
    return None

_ = maybe_extract_zip()

def has_required_files(root: Path):
    required = [
        'dataset/train/train.zh',
        'dataset/train/train.vi',
        'dataset/test/test.zh',
    ]
    return all((root / rel).exists() for rel in required)

def find_data_root():
    candidates = [
        Path('./task2_extracted'),
        Path('/mnt/data/task2_unzipped'),
        Path('./task2'),
        Path('.'),
        Path('/content'),
        Path('/kaggle/working'),
    ]

    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        candidates.append(kaggle_input)
        candidates.extend([p for p in kaggle_input.iterdir() if p.is_dir()])

    for root in candidates:
        if root.exists() and has_required_files(root):
            return root

    # recursive fallback for Kaggle attached datasets
    search_roots = [Path('.'), Path('/kaggle/input'), Path('/kaggle/working')]
    for base in search_roots:
        if not base.exists():
            continue
        for p in base.rglob('train.zh'):
            candidate = p.parent.parent.parent
            if has_required_files(candidate):
                return candidate

    raise FileNotFoundError(
        'Could not locate dataset/. In Kaggle, attach the dataset or zip and keep the standard folder structure.'
    )

DATA_ROOT = find_data_root()
TRAIN_SRC = DATA_ROOT / 'dataset/train/train.zh'
TRAIN_TGT = DATA_ROOT / 'dataset/train/train.vi'
TEST_SRC = DATA_ROOT / 'dataset/test/test.zh'

save_dir = Path(cfg.save_dir)
save_dir.mkdir(parents=True, exist_ok=True)

print('DATA_ROOT =', DATA_ROOT.resolve())
print('SAVE_DIR  =', save_dir.resolve())


In [ ]:

# -----------------------------
# Read data + quick EDA
# -----------------------------

def read_lines(path: Path):
    with open(path, 'r', encoding='utf-8') as f:
        return [line.strip() for line in f if line.strip()]

train_src = read_lines(TRAIN_SRC)
train_tgt = read_lines(TRAIN_TGT)
test_src = read_lines(TEST_SRC)

assert len(train_src) == len(train_tgt), 'Mismatched train files.'

print('Train pairs:', len(train_src))
print('Test size  :', len(test_src))
print()
print('Sample pair:')
print('ZH:', train_src[0])
print('VI:', train_tgt[0])

src_lens = [len(x.split()) for x in train_src]
tgt_lens = [len(x.split()) for x in train_tgt]

stats = pd.DataFrame({
    'split': ['train_src', 'train_tgt'],
    'mean_len': [np.mean(src_lens), np.mean(tgt_lens)],
    'p95_len': [np.percentile(src_lens, 95), np.percentile(tgt_lens, 95)],
    'max_len': [np.max(src_lens), np.max(tgt_lens)],
    'vocab': [len(set(tok for s in train_src for tok in s.split())), len(set(tok for s in train_tgt for tok in s.split()))],
})
stats


In [ ]:

# -----------------------------
# Train / validation split
# -----------------------------

def train_valid_split(src, tgt, valid_ratio=0.08, seed=42):
    n = len(src)
    idx = np.arange(n)
    rng = np.random.default_rng(seed)
    rng.shuffle(idx)
    valid_size = max(1, int(n * valid_ratio))
    valid_idx = idx[:valid_size]
    train_idx = idx[valid_size:]
    tr_src = [src[i] for i in train_idx]
    tr_tgt = [tgt[i] for i in train_idx]
    va_src = [src[i] for i in valid_idx]
    va_tgt = [tgt[i] for i in valid_idx]
    return tr_src, tr_tgt, va_src, va_tgt

tr_src, tr_tgt, va_src, va_tgt = train_valid_split(train_src, train_tgt, cfg.valid_ratio, SEED)

if cfg.smoke_test:
    tr_src, tr_tgt = tr_src[:768], tr_tgt[:768]
    va_src, va_tgt = va_src[:96], va_tgt[:96]
    test_src = test_src[:128]

print('Train:', len(tr_src))
print('Valid:', len(va_src))
print('Test :', len(test_src))


In [ ]:

# -----------------------------
# Vocabulary (word-level)
# -----------------------------

PAD = '<pad>'
UNK = '<unk>'
BOS = '<bos>'
EOS = '<eos>'
SPECIALS = [PAD, UNK, BOS, EOS]
PAD_ID, UNK_ID, BOS_ID, EOS_ID = range(4)

class Vocab:
    def __init__(self, token_lists, min_freq=1):
        counter = Counter(tok for toks in token_lists for tok in toks)
        self.itos = SPECIALS[:]
        for tok, freq in counter.most_common():
            if freq >= min_freq and tok not in SPECIALS:
                self.itos.append(tok)
        self.stoi = {tok: i for i, tok in enumerate(self.itos)}

    def encode(self, tokens, add_bos=False, add_eos=False):
        ids = []
        if add_bos:
            ids.append(BOS_ID)
        ids.extend(self.stoi.get(tok, UNK_ID) for tok in tokens)
        if add_eos:
            ids.append(EOS_ID)
        return ids

    def decode(self, ids, skip_specials=True):
        toks = []
        for idx in ids:
            if idx < 0 or idx >= len(self.itos):
                continue
            tok = self.itos[idx]
            if skip_specials and tok in SPECIALS:
                continue
            toks.append(tok)
        return toks

    def __len__(self):
        return len(self.itos)

tr_src_tok = [s.split()[:cfg.max_src_len] for s in tr_src]
tr_tgt_tok = [s.split()[:cfg.max_tgt_len] for s in tr_tgt]
va_src_tok = [s.split()[:cfg.max_src_len] for s in va_src]
va_tgt_tok = [s.split()[:cfg.max_tgt_len] for s in va_tgt]

src_vocab = Vocab(tr_src_tok, min_freq=cfg.min_freq_src)
tgt_vocab = Vocab(tr_tgt_tok, min_freq=cfg.min_freq_tgt)

print('Source vocab:', len(src_vocab))
print('Target vocab:', len(tgt_vocab))
print('Special token ids:', PAD_ID, UNK_ID, BOS_ID, EOS_ID)


In [ ]:

# -----------------------------
# Translation memory for exact matches
# -----------------------------

memory_table = defaultdict(list)
for s, t in zip(train_src, train_tgt):
    memory_table[s].append(t)
exact_match_map = {s: Counter(ts).most_common(1)[0][0] for s, ts in memory_table.items()}

num_exact_dups = sum(1 for s, ts in memory_table.items() if len(ts) > 1)
num_conflicts = sum(1 for s, ts in memory_table.items() if len(set(ts)) > 1)
print('Unique source sentences:', len(memory_table))
print('Sources with duplicates :', num_exact_dups)
print('Conflicting duplicates   :', num_conflicts)


In [ ]:

# -----------------------------
# Dataset / loader
# -----------------------------

class TranslationDataset(Dataset):
    def __init__(self, src_texts, tgt_texts=None, src_vocab=None, tgt_vocab=None, max_src_len=64, max_tgt_len=64):
        self.src_texts = src_texts
        self.tgt_texts = tgt_texts
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab
        self.max_src_len = max_src_len
        self.max_tgt_len = max_tgt_len

    def __len__(self):
        return len(self.src_texts)

    def __getitem__(self, idx):
        src_tokens = self.src_texts[idx].split()[:self.max_src_len]
        src_ids = self.src_vocab.encode(src_tokens, add_bos=True, add_eos=True)

        item = {
            'src_ids': torch.tensor(src_ids, dtype=torch.long),
            'src_text': self.src_texts[idx],
        }

        if self.tgt_texts is not None:
            tgt_tokens = self.tgt_texts[idx].split()[:self.max_tgt_len]
            tgt_ids = self.tgt_vocab.encode(tgt_tokens, add_bos=True, add_eos=True)
            item['tgt_ids'] = torch.tensor(tgt_ids, dtype=torch.long)
            item['tgt_text'] = self.tgt_texts[idx]

        return item


def pad_sequence(sequences, pad_value=PAD_ID):
    max_len = max(len(x) for x in sequences)
    out = torch.full((len(sequences), max_len), pad_value, dtype=torch.long)
    for i, seq in enumerate(sequences):
        out[i, :len(seq)] = seq
    return out


def collate_fn(batch):
    src_ids = pad_sequence([x['src_ids'] for x in batch], PAD_ID)
    out = {
        'src_ids': src_ids,
        'src_texts': [x['src_text'] for x in batch],
    }
    if 'tgt_ids' in batch[0]:
        tgt_ids = pad_sequence([x['tgt_ids'] for x in batch], PAD_ID)
        out['tgt_ids'] = tgt_ids
        out['tgt_texts'] = [x['tgt_text'] for x in batch]
    return out

train_ds = TranslationDataset(tr_src, tr_tgt, src_vocab, tgt_vocab, cfg.max_src_len, cfg.max_tgt_len)
valid_ds = TranslationDataset(va_src, va_tgt, src_vocab, tgt_vocab, cfg.max_src_len, cfg.max_tgt_len)
test_ds = TranslationDataset(test_src, None, src_vocab, tgt_vocab, cfg.max_src_len, cfg.max_tgt_len)

train_loader = DataLoader(
    train_ds,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    collate_fn=collate_fn,
    pin_memory=USE_AMP,
)
valid_loader = DataLoader(
    valid_ds,
    batch_size=max(32, cfg.batch_size),
    shuffle=False,
    num_workers=cfg.num_workers,
    collate_fn=collate_fn,
    pin_memory=USE_AMP,
)

print('Train batches:', len(train_loader))
print('Valid batches:', len(valid_loader))


In [ ]:

# -----------------------------
# Model
# -----------------------------

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=512):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # [1, max_len, d_model]
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


class Seq2SeqTransformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=256, nhead=8,
                 num_encoder_layers=4, num_decoder_layers=4, dim_feedforward=768, dropout=0.1, pad_id=0):
        super().__init__()
        self.pad_id = pad_id
        self.d_model = d_model

        self.src_embed = nn.Embedding(src_vocab_size, d_model, padding_idx=pad_id)
        self.tgt_embed = nn.Embedding(tgt_vocab_size, d_model, padding_idx=pad_id)
        self.pos_encoder = PositionalEncoding(d_model, dropout=dropout)
        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
        )
        self.generator = nn.Linear(d_model, tgt_vocab_size)

    def make_tgt_mask(self, tgt_len, device):
        return torch.triu(torch.ones(tgt_len, tgt_len, device=device, dtype=torch.bool), diagonal=1)

    def encode(self, src_ids):
        src_key_padding_mask = src_ids.eq(self.pad_id)
        src_emb = self.pos_encoder(self.src_embed(src_ids) * math.sqrt(self.d_model))
        memory = self.transformer.encoder(src_emb, src_key_padding_mask=src_key_padding_mask)
        return memory, src_key_padding_mask

    def decode(self, tgt_ids, memory, src_key_padding_mask=None):
        tgt_key_padding_mask = tgt_ids.eq(self.pad_id)
        tgt_mask = self.make_tgt_mask(tgt_ids.size(1), tgt_ids.device)
        tgt_emb = self.pos_encoder(self.tgt_embed(tgt_ids) * math.sqrt(self.d_model))
        out = self.transformer.decoder(
            tgt=tgt_emb,
            memory=memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=src_key_padding_mask,
        )
        return self.generator(out)

    def forward(self, src_ids, tgt_input_ids):
        memory, src_key_padding_mask = self.encode(src_ids)
        return self.decode(tgt_input_ids, memory, src_key_padding_mask)


model = Seq2SeqTransformer(
    src_vocab_size=len(src_vocab),
    tgt_vocab_size=len(tgt_vocab),
    d_model=cfg.d_model,
    nhead=cfg.nhead,
    num_encoder_layers=cfg.num_encoder_layers,
    num_decoder_layers=cfg.num_decoder_layers,
    dim_feedforward=cfg.dim_feedforward,
    dropout=cfg.dropout,
    pad_id=PAD_ID,
).to(DEVICE)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(model)
print(f'Trainable parameters: {count_parameters(model):,}')


In [ ]:
# -----------------------------
# Training utilities
# -----------------------------

criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID, label_smoothing=cfg.label_smoothing)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay, betas=(0.9, 0.98))
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=cfg.lr,
    epochs=cfg.epochs,
    steps_per_epoch=max(1, len(train_loader)),
    pct_start=0.1,
    div_factor=10,
    final_div_factor=50,
)
autocast_device = 'cuda' if DEVICE.type == 'cuda' else 'cpu'
scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)


def run_train_epoch(model, loader):
    model.train()
    losses = []
    pbar = tqdm(loader, desc='train', leave=False)

    for batch in pbar:
        src_ids = batch['src_ids'].to(DEVICE)
        tgt_ids = batch['tgt_ids'].to(DEVICE)

        tgt_in = tgt_ids[:, :-1]
        tgt_out = tgt_ids[:, 1:]

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(autocast_device, enabled=USE_AMP):
            logits = model(src_ids, tgt_in)
            loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        losses.append(loss.item())
        pbar.set_postfix(loss=f'{np.mean(losses):.4f}', lr=f"{optimizer.param_groups[0]['lr']:.2e}")

    return float(np.mean(losses)) if losses else 0.0


In [ ]:
# -----------------------------
# Decoding utilities
# -----------------------------

def length_penalty(length, alpha=0.7):
    return ((5 + length) / 6) ** alpha


def has_repeat_ngram(token_ids, n):
    if n <= 0 or len(token_ids) < n:
        return False
    seen = set()
    for i in range(len(token_ids) - n + 1):
        ng = tuple(token_ids[i:i+n])
        if ng in seen:
            return True
        seen.add(ng)
    return False


@torch.no_grad()
def beam_search_decode(model, src_text, beam_size=4, max_decode_len=64, min_decode_len=1,
                       alpha=0.7, no_repeat_ngram_size=3):
    # exact-match memory first
    if src_text in exact_match_map:
        return exact_match_map[src_text]

    src_tokens = src_text.split()[:cfg.max_src_len]
    src_ids = src_vocab.encode(src_tokens, add_bos=True, add_eos=True)
    src_tensor = torch.tensor(src_ids, dtype=torch.long, device=DEVICE).unsqueeze(0)

    memory, src_key_padding_mask = model.encode(src_tensor)

    beams = [([BOS_ID], 0.0, False)]  # (token_ids, raw_logprob, finished)
    finished = []

    for step in range(max_decode_len):
        candidates = []

        for seq, score, done in beams:
            if done:
                candidates.append((seq, score, done))
                continue

            tgt_tensor = torch.tensor(seq, dtype=torch.long, device=DEVICE).unsqueeze(0)
            logits = model.decode(tgt_tensor, memory, src_key_padding_mask)[0, -1]
            logits[PAD_ID] = -1e9
            logits[BOS_ID] = -1e9
            logits[UNK_ID] = -1e9
            if step < min_decode_len:
                logits[EOS_ID] = -1e9

            log_probs = F.log_softmax(logits, dim=-1)
            topk_log_probs, topk_ids = torch.topk(log_probs, beam_size)

            for tok_id, tok_logprob in zip(topk_ids.tolist(), topk_log_probs.tolist()):
                new_seq = seq + [tok_id]
                if no_repeat_ngram_size > 0 and has_repeat_ngram(new_seq[1:], no_repeat_ngram_size):
                    continue
                new_done = tok_id == EOS_ID
                candidates.append((new_seq, score + tok_logprob, new_done))

        if not candidates:
            break

        def rank_key(x):
            seq, score, done = x
            length = max(1, len(seq) - 1)
            return score / length_penalty(length, alpha)

        new_beams = sorted(candidates, key=rank_key, reverse=True)[:beam_size]
        if not new_beams:
            break

        beams = new_beams

        for beam in beams:
            if beam[2]:
                finished.append(beam)

        if len(finished) >= beam_size and all(done for _, _, done in beams):
            break

    final_beams = finished if finished else beams

    if not final_beams:
        return exact_match_map.get(src_text, '')

    final_beams = sorted(
        final_beams,
        key=lambda x: x[1] / length_penalty(max(1, len(x[0]) - 1), alpha),
        reverse=True
    )
    best_tokens = final_beams[0][0]

    if EOS_ID in best_tokens[1:]:
        eos_pos = best_tokens[1:].index(EOS_ID) + 1
        best_tokens = best_tokens[1:eos_pos]
    else:
        best_tokens = best_tokens[1:]

    pred_tokens = tgt_vocab.decode(best_tokens, skip_specials=True)
    return ' '.join(pred_tokens).strip()


In [ ]:
# -----------------------------
# Validation BLEU
# -----------------------------

@torch.no_grad()
def predict_texts(model, src_texts, show_progress=True):
    preds = []
    iterator = tqdm(src_texts, desc='decode', leave=False) if show_progress else src_texts
    for src in iterator:
        pred = beam_search_decode(
            model,
            src,
            beam_size=cfg.beam_size,
            max_decode_len=cfg.max_decode_len,
            min_decode_len=cfg.min_decode_len,
            alpha=cfg.length_penalty_alpha,
            no_repeat_ngram_size=cfg.no_repeat_ngram_size,
        )
        preds.append(pred)
    return preds


@torch.no_grad()
def evaluate_bleu(model, valid_src, valid_tgt):
    model.eval()
    preds = predict_texts(model, valid_src, show_progress=True)
    if sacrebleu is not None:
        bleu = sacrebleu.corpus_bleu(preds, [valid_tgt], force=True, tokenize='none').score
    else:
        bleu = fallback_corpus_bleu(preds, valid_tgt)
        print('Warning: using fallback BLEU approximation because sacrebleu is unavailable.')
    return bleu, preds


In [ ]:

# -----------------------------
# Train loop with checkpointing
# -----------------------------

best_bleu = -1.0
best_path = save_dir / f'{cfg.run_name}_best.pt'
history = []
patience = 0

for epoch in range(1, cfg.epochs + 1):
    t0 = time.time()
    train_loss = run_train_epoch(model, train_loader)
    val_bleu, val_preds = evaluate_bleu(model, va_src, va_tgt)
    elapsed = time.time() - t0

    row = {
        'epoch': epoch,
        'train_loss': train_loss,
        'val_bleu': val_bleu,
        'minutes': elapsed / 60,
    }
    history.append(row)
    print(f"Epoch {epoch:02d} | loss={train_loss:.4f} | val_bleu={val_bleu:.2f} | time={elapsed/60:.1f} min")

    if val_bleu > best_bleu:
        best_bleu = val_bleu
        patience = 0
        torch.save({
            'model_state_dict': model.state_dict(),
            'src_vocab': src_vocab.itos,
            'tgt_vocab': tgt_vocab.itos,
            'cfg': asdict(cfg),
            'best_bleu': best_bleu,
            'epoch': epoch,
        }, best_path)
        print(f'  -> saved best checkpoint to {best_path}')
    else:
        patience += 1
        if patience >= cfg.early_stopping:
            print('Early stopping triggered.')
            break

history_df = pd.DataFrame(history)
history_df


In [ ]:

# -----------------------------
# Load best checkpoint + inspect samples
# -----------------------------

ckpt = torch.load(best_path, map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print('Loaded best checkpoint from epoch', ckpt['epoch'])
print('Best validation BLEU:', round(ckpt['best_bleu'], 4))

sample_count = min(12, len(va_src))
sample_preds = predict_texts(model, va_src[:sample_count], show_progress=False)
sample_df = pd.DataFrame({
    'source_zh': va_src[:sample_count],
    'reference_vi': va_tgt[:sample_count],
    'prediction_vi': sample_preds,
})
sample_df


In [ ]:

# -----------------------------
# Predict test set
# -----------------------------

test_preds = predict_texts(model, test_src, show_progress=True)
submission_df = pd.DataFrame({
    'tieng_trung': test_src,
    'tieng_viet': test_preds,
})
submission_df.head(10)


In [ ]:

# -----------------------------
# Save submission.csv + submission.zip
# -----------------------------

submission_csv = save_dir / 'submission.csv'
submission_zip = save_dir / 'submission.zip'

submission_df.to_csv(submission_csv, index=False, encoding='utf-8-sig')
with zipfile.ZipFile(submission_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(submission_csv, arcname='submission.csv')

print('Saved CSV:', submission_csv.resolve())
print('Saved ZIP:', submission_zip.resolve())
print('ZIP size :', f'{submission_zip.stat().st_size / 1024:.2f} KB')



## Practical tuning ideas

If you want to push the score further, the first knobs to try are:

1. increase `epochs` to 25-30 if validation BLEU is still climbing
2. try `d_model=384`, `num_encoder_layers=5`, `num_decoder_layers=5` on a GPU
3. try `beam_size=5` or `6`
4. increase `valid_ratio` stability by training a few seeds and keeping the best checkpoint
5. add checkpoint averaging across the best 2-3 epochs

For leaderboard submissions, always keep the exact output column names:

- `tieng_trung`
- `tieng_viet`
